# Notebook 05: Context Management Pattern

**CCA Pattern:** Structured Summaries vs Raw Transcript

This notebook runs the same 5-turn conversation twice, once with each context strategy, and sends the accumulated context to the model on every turn:

1. **Anti-Pattern**: Raw transcript accumulator — the context sent grows O(n) with every turn
2. **Correct Pattern**: `ContextSummary` — a fixed-field schema whose rendered size is bounded
3. **Compare**: Context tokens sent per turn, API input tokens per turn, and whether the deadline from Turn 1 is recalled at Turn 5

## Setup

Install dependencies and set up services. Requires `ANTHROPIC_API_KEY` environment variable.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

import anthropic
from helpers import compare_results
from tabulate import tabulate

from customer_service.agent.agent_loop import AgentResult, run_agent_loop
from customer_service.agent.context_manager import TOKEN_BUDGET, ContextSummary
from customer_service.agent.system_prompts import get_system_prompt
from customer_service.anti_patterns.raw_transcript import RawTranscriptContext
from customer_service.data.customers import CUSTOMERS
from customer_service.services.audit_log import AuditLog
from customer_service.services.container import ServiceContainer
from customer_service.services.customer_db import CustomerDatabase
from customer_service.services.escalation_queue import EscalationQueue
from customer_service.services.financial_system import FinancialSystem
from customer_service.services.policy_engine import PolicyEngine

In [ ]:
def make_services() -> ServiceContainer:
    return ServiceContainer(
        customer_db=CustomerDatabase(CUSTOMERS),
        policy_engine=PolicyEngine(),
        financial_system=FinancialSystem(),
        escalation_queue=EscalationQueue(),
        audit_log=AuditLog(),
    )


from dotenv import find_dotenv, load_dotenv

# Load ANTHROPIC_API_KEY from .env (find_dotenv walks up from the notebooks/ dir)
load_dotenv(find_dotenv(), override=False)

client = anthropic.Anthropic()

## The Problem: Context Growth and the Lost-in-Middle Effect

An agent that keeps a session going needs to carry context from one turn to the next. Two ways to do that:

- **Raw transcript**: append every user message, assistant reply and tool call as text, and send all of it on every turn. The context sent grows **O(n)** with the number of turns. Over a long session, early facts sit in the middle of a large block of text, which is where models attend least (the lost-in-middle effect).
- **Structured summary**: keep a fixed-field record (customer, issue, recent tools, recent decisions, pending actions) and send only that. Its rendered size is **bounded** no matter how many turns have passed.

> **CCA Exam fact:** A bigger context window makes the lost-in-middle effect **WORSE**, not better. More context = more dilution, not less.

**What this notebook can and cannot show.** A 5-turn support conversation is a few thousand characters. A current model does not lose a fact at that scale, so both strategies will recall the Turn 1 deadline at Turn 5, and the notebook checks that honestly. What differs, and what the cells measure, is **how much context each strategy sends on every turn** and therefore what every turn costs. That difference is deterministic and it is the reason the pattern exists.

In [ ]:
# The same 5-turn conversation is used for both patterns.
# Turn 1 states the customer ID, the order, the amount, and a deadline.
# Turns 2-5 never repeat the ID: the agent must get it from the context it is sent.
# Turn 5 asks for the deadline back, so recall can be checked in the reply.
DEADLINE = "March 30"

user_messages = [
    "I need help with order #ORD-001. It was a $50 birthday gift for my mother "
    "and the item arrived broken. Her birthday is March 30th — I really need "
    "this resolved before then. My customer ID is C001.",
    "What is the refund policy for defective items? Will I get a full refund or just store credit?",
    "I can confirm the item was defective right out of the box — it would not "
    "turn on at all. I have photos of the damage if needed.",
    "How long does the refund take to process? Will it show up on my card within a few days?",
    "Can you confirm that the refund has been processed, summarize what was resolved "
    "in this session, and remind me of the deadline I mentioned?",
]

print(f"Conversation: {len(user_messages)} turns")
print("Customer: C001 (Alice Johnson, Regular tier)")
print(f"Key early fact (Turn 1): birthday gift, {DEADLINE} deadline")


def api_calls(result: AgentResult) -> int:
    """One API call per assistant message in the loop's transcript."""
    return sum(1 for m in result.messages if m["role"] == "assistant")

## Anti-Pattern: Raw Transcript Accumulator

<div style="background-color:#fff0f0; border-left:4px solid #cc0000; padding:16px; margin:12px 0; border-radius:4px;">

**WRONG APPROACH: Append every message and tool call to a raw text string, send all of it every turn**

`RawTranscriptContext` appends every user message, every tool call and every assistant reply as raw text. On each turn the whole transcript is appended to the system prompt. Nothing is ever dropped, so the context sent grows with every turn.

</div>

One service container is shared across the 5 turns, as it would be in a real session. Each row below records how many context tokens were **sent** on that turn (the transcript as it stood before the turn) and what the API charged for the turn.

In [ ]:
raw_transcript = RawTranscriptContext()
services_raw = make_services()
raw_rows: list[dict] = []

for turn, message in enumerate(user_messages, start=1):
    context = raw_transcript.to_context_string()
    system = get_system_prompt() + "\n\n" + context  # whole transcript, every turn
    result = run_agent_loop(client, services_raw, message, system)

    raw_transcript.append("user", message)
    for call in result.tool_calls:
        raw_transcript.append("tool", f"{call['name']}({json.dumps(call['input'])})")
    raw_transcript.append("assistant", result.final_text or "(no text response)")

    raw_rows.append(
        {
            "turn": turn,
            "context_tokens_sent": len(context) // 4,
            "api_calls": api_calls(result),
            "input_tokens": result.usage.input_tokens,
            "output_tokens": result.usage.output_tokens,
            "final_text": result.final_text,
        }
    )
    print(
        f"Turn {turn}: sent {raw_rows[-1]['context_tokens_sent']:>5} context tokens | "
        f"{raw_rows[-1]['api_calls']} API calls | "
        f"{raw_rows[-1]['input_tokens']:>6,} input tokens"
    )

print(f"\nTranscript after 5 turns: {raw_transcript.token_estimate()} tokens (estimate)")
print(f"Deadline recalled at Turn 5: {DEADLINE in raw_rows[-1]['final_text']}")
print(f"\nTurn 5 reply:\n{raw_rows[-1]['final_text']}")

## Correct Pattern: Structured Context Summary

<div style="background-color:#f0fff0; border-left:4px solid #00aa00; padding:16px; margin:12px 0; border-radius:4px;">

**CORRECT APPROACH: `ContextSummary` with a fixed-field schema, rendered and sent every turn**

`ContextSummary` holds `customer_id`, `issue_type`, `tools_called`, `decisions_made` and `pending_actions`. Its `to_system_context()` renders only the last 5 tools and last 3 decisions, and `update()` compacts the lists if the rendered size exceeds `TOKEN_BUDGET`. So the context sent each turn stays small however long the session runs.

</div>

**Filling the summary is the real work of this pattern**, and the notebook makes it explicit:

- After each turn, every tool call the agent made is recorded with `update()`. The one-line description is built from the tool's name and inputs by a small function below; in production that step is code or a dedicated extraction call, never the raw text.
- The deadline is a fact from the customer's message, not a tool call. Here it is placed in `pending_actions` by hand before Turn 1, which is the honest way to say: **somebody has to extract it**. Facts do not get into a structured summary on their own.

Note the budget units: `TOKEN_BUDGET` is compared against `token_estimate`, which is characters divided by 4, so it is about 300 tokens or 1,200 characters of rendered context.

In [ ]:
def describe_tool_call(call: dict) -> str:
    """One short line per tool call, built from name and inputs (never the raw result)."""
    args = ", ".join(f"{k}={v}" for k, v in call["input"].items() if k != "customer_id")
    return f"{call['name']}({args})"[:80]


summary = ContextSummary()
summary.customer_id = "C001"
summary.issue_type = "defective_item_refund"
summary.pending_actions.append(f"birthday gift deadline: {DEADLINE}th")  # extracted by hand

print(f"TOKEN_BUDGET: {TOKEN_BUDGET} estimated tokens (~{TOKEN_BUDGET * 4} chars)")
print("Initial structured context:")
print(summary.to_system_context())
print()

services_summary = make_services()
summary_rows: list[dict] = []

for turn, message in enumerate(user_messages, start=1):
    context = summary.to_system_context()
    system = get_system_prompt() + "\n\n" + context  # bounded summary, every turn
    result = run_agent_loop(client, services_summary, message, system)

    for call in result.tool_calls:
        summary.update(call["name"], describe_tool_call(call))

    summary_rows.append(
        {
            "turn": turn,
            "context_tokens_sent": len(context) // 4,
            "api_calls": api_calls(result),
            "input_tokens": result.usage.input_tokens,
            "output_tokens": result.usage.output_tokens,
            "final_text": result.final_text,
        }
    )
    print(
        f"Turn {turn}: sent {summary_rows[-1]['context_tokens_sent']:>5} context tokens | "
        f"{summary_rows[-1]['api_calls']} API calls | "
        f"{summary_rows[-1]['input_tokens']:>6,} input tokens"
    )

print(f"\nSummary after 5 turns: {summary.token_estimate} tokens (estimate), budget {TOKEN_BUDGET}")
print(f"Deadline recalled at Turn 5: {DEADLINE in summary_rows[-1]['final_text']}")
print(f"\nFinal structured context:\n{summary.to_system_context()}")
print(f"\nTurn 5 reply:\n{summary_rows[-1]['final_text']}")

### The bound, without an API call

Five turns are not enough to make `ContextSummary` compact anything; its rendered size stays well under budget on its own. To see the bound hold, push 30 long entries through both structures. The raw transcript grows with every entry. The summary compacts, stays under `TOKEN_BUDGET`, and keeps `customer_id`, `issue_type` and the pending deadline.

One constraint to know: compaction drops **whole entries** (it keeps the last 2 decisions, then the last pending action). It never shortens an entry. So two entries longer than about half the budget cannot be compacted under it, which is why `describe_tool_call` above caps each entry at 80 characters. Keep entries short; the schema does the rest.

In [ ]:
stress_raw = RawTranscriptContext()
stress_summary = ContextSummary()
stress_summary.customer_id = "C001"
stress_summary.issue_type = "defective_item_refund"
stress_summary.pending_actions.append(f"birthday gift deadline: {DEADLINE}th")

peak_summary = 0
for i in range(30):
    entry = f"decision {i}: " + "policy detail " * 28  # ~400 chars, like a verbose tool result
    stress_raw.append("tool", entry)
    stress_summary.update("check_policy", entry)
    peak_summary = max(peak_summary, stress_summary.token_estimate)

print(f"After 30 entries — raw transcript: {stress_raw.token_estimate():,} tokens (grows with n)")
print(f"After 30 entries — summary: {stress_summary.token_estimate} tokens, peak {peak_summary}")
print(f"Summary always under budget ({TOKEN_BUDGET}): {peak_summary <= TOKEN_BUDGET}")
print(f"customer_id kept: {stress_summary.customer_id == 'C001'}")
print(f"deadline kept in pending_actions: {DEADLINE in str(stress_summary.pending_actions)}")
print(f"decisions retained after compaction: {len(stress_summary.decisions_made)} of 30")

## Compare Results

Per-turn, side by side. `context tokens` is what each strategy added to the system prompt on that turn (the accumulated context as it stood **before** the turn). `input tokens` is what the API charged for the turn, which also includes the tools, the instructions and any tool round-trips, so it moves with how many calls the agent made.

Then a direct comparison after 5 turns. Both strategies recall the deadline: at this length nothing is lost, and the table says so. The context gap at Turn 5 is about 90%, but the total-cost gap is small, because at 5 turns the tools and instructions still dominate every call. The raw context grows by a few hundred tokens per turn and never stops; by turn 30 or 50 it is the largest thing in every request. The pattern's value is that growth rate, not a saving you can see in five turns.

In [ ]:
headers = ["Turn", "Raw ctx tokens", "Raw input tokens", "Summary ctx tokens", "Summary input tokens"]
rows = [
    [r["turn"], r["context_tokens_sent"], r["input_tokens"], s["context_tokens_sent"], s["input_tokens"]]
    for r, s in zip(raw_rows, summary_rows, strict=True)
]
print(tabulate(rows, headers=headers, tablefmt="simple"))
print()

compare_results(
    {
        "context_tokens_sent_turn5": raw_rows[-1]["context_tokens_sent"],
        "total_input_tokens_5_turns": sum(r["input_tokens"] for r in raw_rows),
        "total_api_calls": sum(r["api_calls"] for r in raw_rows),
        "deadline_recalled_turn5": DEADLINE in raw_rows[-1]["final_text"],
    },
    {
        "context_tokens_sent_turn5": summary_rows[-1]["context_tokens_sent"],
        "total_input_tokens_5_turns": sum(s["input_tokens"] for s in summary_rows),
        "total_api_calls": sum(s["api_calls"] for s in summary_rows),
        "deadline_recalled_turn5": DEADLINE in summary_rows[-1]["final_text"],
    },
)

## CCA Exam Tip: Context Management

> **CCA Exam Tip:**
> 
> **The Lost-in-Middle Effect:**
> - Raw transcripts cause O(n) token growth — each turn makes the context longer
> - Important facts from early turns get buried under later content in long sessions
> - The model's attention focuses on recent content; early facts are diluted
> - **Bigger context window makes it WORSE, not better** — more dilution, not less
> 
> **Correct remedy:** `ContextSummary` with fixed-field schema:
> - `customer_id`, `issue_type` — structured fields that survive compaction
> - `decisions_made`, `pending_actions` — short bullets, bounded by `TOKEN_BUDGET`
> - `token_estimate` — stays bounded regardless of turn count
> - Someone has to **extract** facts into the fields — code or a dedicated extraction step, never the raw text
> 
> **Exam signal → Answer:**
> - "Quality drops in longer sessions" → Context degradation → use structured summaries
> - "Important facts forgotten by turn 5" → Lost-in-middle → use `ContextSummary`
> - "Use a bigger context window" → TRAP — makes it worse

## Summary

| Metric | Anti-Pattern (Raw) | Correct (ContextSummary) |
|--------|-------------------|-------------------------|
| Context sent per turn | O(n) — grows every turn | O(1) — bounded at TOKEN_BUDGET |
| Early facts | Present, but buried deeper each turn | In named fields, never buried |
| Compaction | None — grows forever | Automatic, structured fields survive |
| API cost | Grows each turn | Flat (bounded context injection) |
| Deadline recalled at 5 turns | Yes | Yes — the gap is cost and scale, not recall |

**Key files:**

- `src/customer_service/agent/context_manager.py` — `ContextSummary`, `TOKEN_BUDGET`
- `src/customer_service/anti_patterns/raw_transcript.py` — `RawTranscriptContext`

**CCA Rule:** 'Quality drops in longer sessions? → Context degradation → Use `ContextSummary` with fixed-field schema, not raw transcripts.'